# Notebook 02: Train Modified ConvNeXt

This notebook walks through training the Modified ConvNeXt with ECB on a single fold of OncoLung60K.

For full 5-fold CV, run from the command line:
```bash
python -m src.kfold_cv --config configs/oncolung_modified_convnext.yaml \
    --data_dir data/oncolung60k --output_dir runs/cv_modified
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import yaml
import torch
import pandas as pd
from torch.utils.data import DataLoader

from src.data.augmentation import build_transforms
from src.data.dataset import HistoPathologyDataset
from src.models.builder import build_model
from src.train import train_one_epoch, evaluate, set_seed

## Load configuration

In [ ]:
cfg = yaml.safe_load(Path('../configs/oncolung_modified_convnext.yaml').read_text())
cfg

## Set up data

In [ ]:
DATA_ROOT = '../data/oncolung60k'  # adjust as needed
TEST_FOLD = 0

df = pd.read_csv('..' / Path(cfg['splits_csv']))
train_df = df[df['fold'] != TEST_FOLD].reset_index(drop=True)
test_df  = df[df['fold'] == TEST_FOLD].reset_index(drop=True)
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

train_tf = build_transforms(cfg['image_size'], is_training=True)
eval_tf  = build_transforms(cfg['image_size'], is_training=False)
train_set = HistoPathologyDataset(train_df, DATA_ROOT, transform=train_tf)
test_set  = HistoPathologyDataset(test_df,  DATA_ROOT, transform=eval_tf)

train_loader = DataLoader(train_set, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=4, pin_memory=True)

## Build model

In [ ]:
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model('modified_convnext', num_classes=4, pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {n_params:.1f}M')
print(f'Device:     {device}')

## Training loop

In [ ]:
import torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.1))
scaler = torch.cuda.amp.GradScaler() if device == 'cuda' else None

history = []
best_acc = 0.0
for epoch in range(20):  # short demo run; full training uses 100
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    scheduler.step()
    metrics = evaluate(model, test_loader, device, 4)
    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc, **{k:v for k,v in metrics.items() if isinstance(v,(int,float))}})
    print(f"Ep {epoch+1:3d} | tr_acc {tr_acc:.4f} | val_acc {metrics['accuracy']:.4f} | val_auc {metrics['auc_macro_ovr']:.4f}")
    if metrics['accuracy'] > best_acc:
        best_acc = metrics['accuracy']
        torch.save(model.state_dict(), 'best.pt')
print(f'\nBest val accuracy: {best_acc:.4f}')

## Plot training curves

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h['epoch'], h['tr_acc'], label='Train'); ax[0].plot(h['epoch'], h['accuracy'], label='Val')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(h['epoch'], h['auc_macro_ovr']); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Macro AUC')
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()